# BiLSTM Incremental Learning trên ATLASv2 (Domain-Incremental Learning)

Mô hình BiLSTM (Phương án B - 5 đặc trưng: app_family, proto, resp_port_class, image_dir_class, delta_t_log1p).
Huấn luyện tăng tiến (Continual Learning) từ checkpoint đã học trên DARPA OpTC sang miền dữ liệu mới ATLASv2.
Áp dụng bộ đệm Experience Replay (ER) từ OpTC TRAIN để chống Catastrophic Forgetting.
Tập dữ liệu ATLASv2 được phân chia 60% Train / 20% Val / 20% Test theo kịch bản chuẩn tắc.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Cấu hình
Tên run cố định: `bilstm_v2_subsetB`.

In [ ]:
RUN_NAME = "bilstm_atlasv2_incremental"
print("RUN_NAME =", RUN_NAME)


In [ ]:
# Auto-extract portable archive if input.tar is attached on Kaggle and not yet extracted
from pathlib import Path
import tarfile

target = Path("/kaggle/working/optc_network_dl")
already_extracted = any(Path("/kaggle/input").rglob("dataset_manifest.json")) if Path("/kaggle/input").exists() else False

if not already_extracted and not (target / "dataset_manifest.json").exists():
    archives = list(Path("/kaggle/input").rglob("*.tar")) if Path("/kaggle/input").exists() else []
    if archives:
        target.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archives[0]) as archive:
            archive.extractall(target)
        print(f"Extracted {archives[0]} to {target}")
elif (target / "dataset_manifest.json").exists():
    print(f"Dataset already extracted in {target}")
else:
    print("Dataset already extracted in /kaggle/input; skipping archive extraction.")


## Resolve dataset
Notebook xử lý cả trường hợp nested folders được mount trực tiếp hoặc Kaggle CLI đã zip top-level directories.


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    working_root = Path(working_root)
    # Check if already extracted into working_root (e.g. from input.tar)
    if (working_root / "dataset_manifest.json").exists() and (working_root / "build_config.json").exists():
        print(f"Using already extracted dataset root at: {working_root}")
        return working_root

    # Direct official dataset candidates
    official_candidates = [
        Path("/kaggle/input/datasets/ngkimngn/dl-il-atlas"),
        Path("/kaggle/input/dl-il-atlas"),
    ]
    source_root = None
    for cand in official_candidates:
        if (cand / "dataset_manifest.json").exists() and (cand / "build_config.json").exists():
            source_root = cand
            print(f"Matched official dataset path: {source_root}")
            break

    # If not in official candidates, search recursively
    if source_root is None:
        input_root = Path(input_root)
        manifests = list(input_root.rglob("dataset_manifest.json"))
        if not manifests:
            raise FileNotFoundError(
                "Cannot find dataset_manifest.json under /kaggle/input or /kaggle/working/optc_network_dl"
            )
        for manifest in manifests:
            candidate = manifest.parent
            if (candidate / "build_config.json").exists():
                source_root = candidate
                break
        if source_root is None:
            source_root = manifests[0].parent

    print(f"Selected source dataset root: {source_root}")

    if working_root.exists():
        shutil.rmtree(working_root)
    working_root.mkdir(parents=True, exist_ok=True)

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
        "optc_replay_buffer.npz",
    ):
        src = source_root / filename
        if src.exists():
            (working_root / filename).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
        "pretrained_optc_model",
        "optc_test",
    ):
        target = source_root / folder
        if target.exists():
            real_folder = unwrap_same_name_folder(source_root, folder) if target.is_dir() else target
            (working_root / folder).symlink_to(real_folder, target_is_directory=True)

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative
        print(f"{'PASS' if path.exists() else 'FAIL':4s}", relative)
        if not path.exists():
            raise FileNotFoundError(path)

    for optional_item in ("optc_replay_buffer.npz", "optc_test", "pretrained_optc_model"):
        p = working_root / optional_item
        print(f"{'OPT-PASS' if p.exists() else 'OPT-SKIP':8s}", optional_item)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


## Chọn field đầu vào (phương án B)
Ghi `preprocessing/feature_subset.json` và sửa spec trong bản làm việc. Không đụng dataset gốc.

In [ ]:
import shutil
KEEP_CAT = ['app_family', 'proto', 'resp_port_class', 'image_dir_class']   # phương án B (STEP 23): bỏ service, conn_state, principal_class, 7 numeric Bro, has_missed_bytes
pre = DATASET_ROOT / 'preprocessing'
if pre.is_symlink():  # trên Kaggle thư mục này là symlink tới /kaggle/input (chỉ đọc) nên chép ra bản ghi được
    real = pre.resolve(); pre.unlink(); shutil.copytree(real, pre)
spec_path = pre / 'network_preprocess_spec.json'
spec = json.loads(spec_path.read_text(encoding='utf-8'))
layout = json.loads((pre / 'preprocess_spec.json').read_text(encoding='utf-8'))['model_input_layout']
all_cat = list(layout['x_cat_columns'])  # thứ tự cột x_cat trong shards (chạy lại cell này vẫn an toàn)
assert all(f in all_cat for f in KEEP_CAT), KEEP_CAT
spec['active_categorical_features'] = list(KEEP_CAT)
spec['active_numerical_features'] = []
spec['feature_subset_note'] = 'STEP 23 phuong an B: chi giu ' + ', '.join(KEEP_CAT) + ' + delta_t_log1p'
spec_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False), encoding='utf-8')
(pre / 'feature_subset.json').write_text(json.dumps({
    'x_cat_columns_all': all_cat, 'keep_categorical': list(KEEP_CAT), 'drop_numeric': True, 'zero_binary': True,
}, indent=2), encoding='utf-8')
print('categorical giu:', KEEP_CAT)
print('numeric: bo het (0 cot); has_missed_bytes: hang 0; delta_t_log1p: giu')


## Write versioned training modules into `/kaggle/working`


In [ ]:
WORK_CODE = Path('/kaggle/working/step05b_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)

(WORK_CODE / 'bilstm.py').write_text('#!/usr/bin/env python3\n"""BiLSTM Network DL Baseline Architecture.\n\nMatches exactly the model architecture and layer weights from DARPA OpTC\n(06d_BiLSTM_Kaggle_v2_final.ipynb) for seamless transfer and domain-incremental learning.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Any\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence\n\n\ndef embedding_dim(cardinality: int) -> int:\n    """Calculate embedding dimension proportional to square root of cardinality."""\n    return int(min(16, max(4, round(math.sqrt(cardinality) * 2))))\n\n\nclass BiLSTMBaseline(nn.Module):\n    """Bidirectional LSTM model over sliding window sequences of network flow events."""\n\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int = 0,\n        binary_dim: int = 1,\n        event_dim: int = 64,\n        hidden_dim: int = 64,\n        lstm_layers: int = 1,\n        dropout: float = 0.2,\n    ) -> None:\n        super().__init__()\n        self.categorical_cardinalities = categorical_cardinalities\n        self.numerical_dim = numerical_dim\n        self.binary_dim = binary_dim\n        self.event_dim = event_dim\n        self.hidden_dim = hidden_dim\n        self.lstm_layers = lstm_layers\n\n        # Categorical embeddings\n        self.embeddings = nn.ModuleList([\n            nn.Embedding(cardinality, embedding_dim(cardinality))\n            for cardinality in categorical_cardinalities\n        ])\n        cat_dim = sum(emb.embedding_dim for emb in self.embeddings)\n\n        # Raw event dimension: embeddings + numeric + missing indicators + binary + delta_t\n        raw_event_dim = cat_dim + numerical_dim + numerical_dim + binary_dim + 1\n\n        # Event encoder\n        self.event_encoder = nn.Sequential(\n            nn.Linear(raw_event_dim, event_dim),\n            nn.LayerNorm(event_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n        # Bidirectional LSTM\n        self.lstm = nn.LSTM(\n            input_size=event_dim,\n            hidden_size=hidden_dim,\n            num_layers=lstm_layers,\n            batch_first=True,\n            bidirectional=True,\n            dropout=dropout if lstm_layers > 1 else 0.0,\n        )\n\n        # Classification head with pooled bidirectional states\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(hidden_dim * 2),\n            nn.Linear(hidden_dim * 2, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        lengths: torch.Tensor,\n        attention_mask: torch.Tensor | None = None,\n        return_features: bool = False,\n        **kwargs: Any,\n    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:\n        """Forward pass over a batch of variable-length event sequences.\n\n        Incorporates Anti-Dilution Pooling (Temp/task/4.md Action 3): combines final hidden\n        states with time-dimension max-pooling to retain peak attack activations (e.g., C2 beacons,\n        payload execution) even across long dormant windows without adding new parameters.\n\n        Args:\n            x_cat: (batch_size, seq_len, num_cats) categorical integer IDs\n            x_num: (batch_size, seq_len, num_numerics) continuous features\n            x_num_missing: (batch_size, seq_len, num_numerics) missing flags\n            x_bin: (batch_size, seq_len, num_binaries) binary features\n            delta_t_log1p: (batch_size, seq_len, 1) log1p(delta_t)\n            lengths: (batch_size,) integer length of each sequence\n            return_features: if True, returns tuple (logits, pooled_representation)\n\n        Returns:\n            logits: (batch_size,) raw scalar prediction before sigmoid, or (logits, pooled)\n        """\n        cat_parts = [embedding(x_cat[:, :, i]) for i, embedding in enumerate(self.embeddings)]\n        cat_concat = torch.cat(cat_parts, dim=-1)\n        # Embedding Dropout (Run 8 / Task 5.md Thay đổi 3): prevents memorizing individual process tokens (e.g. firefox.exe)\n        cat_concat = F.dropout(cat_concat, p=0.15, training=self.training)\n        event = torch.cat([cat_concat, x_num, x_num_missing, x_bin, delta_t_log1p], dim=-1)\n        event = self.event_encoder(event)\n\n        # Pack sequence for LSTM computation ignoring padding\n        packed = pack_padded_sequence(\n            event,\n            lengths.cpu(),\n            batch_first=True,\n            enforce_sorted=False,\n        )\n        lstm_out_packed, (h_n, _) = self.lstm(packed)\n        lstm_out, _ = pad_packed_sequence(lstm_out_packed, batch_first=True)\n\n        # 1. Final hidden state of forward and backward LSTM: (batch_size, hidden_dim * 2)\n        h_last = torch.cat([h_n[-2], h_n[-1]], dim=-1)\n\n        # 2. Top-K Mean Pooling (k=15) over valid time dimension (Run 8 / Task 5.md Thay đổi 1)\n        # Smooths out single-event outlier spikes while capturing sustained anomaly amplitude\n        seq_mask = torch.arange(lstm_out.size(1), device=lengths.device)[None, :] < lengths[:, None]\n        fill_val = -1e4 if lstm_out.dtype in (torch.float16, torch.bfloat16) else -1e9\n        lstm_out_masked = lstm_out.masked_fill(~seq_mask.unsqueeze(-1), fill_val)\n\n        k_steps = min(15, int(lstm_out.size(1)))\n        topk_vals, _ = torch.topk(lstm_out_masked, k=k_steps, dim=1)  # (batch_size, k_steps, hidden_dim * 2)\n\n        # Valid step mask for sequences shorter than k_steps (avoids averaging padding fill_val)\n        valid_k = torch.clamp(lengths, max=k_steps).unsqueeze(1).unsqueeze(2)  # (batch_size, 1, 1)\n        k_idx = torch.arange(k_steps, device=lengths.device).unsqueeze(0).unsqueeze(2)  # (1, k_steps, 1)\n        valid_mask = k_idx < valid_k  # (batch_size, k_steps, 1)\n\n        topk_vals_clean = topk_vals.masked_fill(~valid_mask, 0.0)\n        h_topk_mean = topk_vals_clean.sum(dim=1) / valid_k.squeeze(1).float().clamp(min=1.0)\n\n        # Run 8 Hybrid Pooling: context preservation + smoothed sustained anomaly activation\n        pooled = 0.5 * h_last + 0.5 * h_topk_mean\n\n        # Output logit\n        logits = self.classifier(pooled).squeeze(-1)\n        if return_features:\n            return logits, pooled\n        return logits\n\n', encoding='utf-8')
(WORK_CODE / 'dataset.py').write_text('#!/usr/bin/env python3\n"""Dataset and DataLoader utilities for Network DL BiLSTM on sequence window shards.\n\nMatches the OpTC v2 pipeline specifications with memory-efficient LRU shard caching\nand shard-aware batch sampling.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport random\nfrom collections import OrderedDict, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\n\nclass PackedNetworkDataset(Dataset):\n    """Memory-efficient Dataset reading sliding-window sequences from compressed .npz shards."""\n\n    def __init__(\n        self,\n        dataset_root: str | Path,\n        split: str,\n        *,\n        cache_shards: int = 2,\n    ) -> None:\n        self.dataset_root = Path(dataset_root)\n        self.split = split.lower().strip()\n        self.cache_shards = max(1, int(cache_shards))\n        self.cat_idx = None\n        self.drop_numeric = False\n        self.zero_binary = False\n\n        subset_file = self.dataset_root / "preprocessing" / "feature_subset.json"\n        if subset_file.exists():\n            cfg = json.loads(subset_file.read_text(encoding="utf-8"))\n            order = cfg.get("x_cat_columns_all", [])\n            keep = cfg.get("keep_categorical", order)\n            if order and keep:\n                self.cat_idx = [order.index(f) for f in keep if f in order]\n            self.drop_numeric = bool(cfg.get("drop_numeric", False))\n            self.zero_binary = bool(cfg.get("zero_binary", False))\n\n        if self.split not in {"train", "val", "test"}:\n            raise ValueError(f"Split must be train/val/test, got \'{self.split}\'")\n\n        manifest = self.dataset_root / "split" / "network_sample_manifest.csv"\n        if not manifest.exists():\n            raise FileNotFoundError(f"Manifest not found: {manifest}")\n\n        self.rows: list[dict[str, Any]] = []\n        with manifest.open("r", newline="", encoding="utf-8") as f:\n            for row in csv.DictReader(f):\n                if row["split"] != self.split:\n                    continue\n                self.rows.append({\n                    "sample_id": row["sample_id"],\n                    "label": int(row["label"]),\n                    "event_count": int(row["event_count"]),\n                    "shard_file": row["shard_file"],\n                    "shard_sample_index": int(row["shard_sample_index"]),\n                })\n\n        if not self.rows:\n            raise RuntimeError(f"No samples found in {manifest} for split=\'{self.split}\'")\n\n        self._cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def _load_shard(self, relative_path: str) -> dict[str, np.ndarray]:\n        cached = self._cache.get(relative_path)\n        if cached is not None:\n            self._cache.move_to_end(relative_path)\n            return cached\n\n        path = self.dataset_root / relative_path\n        if not path.exists():\n            raise FileNotFoundError(f"Shard file not found: {path}")\n\n        with np.load(path, allow_pickle=False) as z:\n            shard = {key: z[key] for key in z.files}\n\n        self._cache[relative_path] = shard\n        self._cache.move_to_end(relative_path)\n        while len(self._cache) > self.cache_shards:\n            self._cache.popitem(last=False)\n        return shard\n\n    def _x_cat(self, x: np.ndarray) -> np.ndarray:\n        if self.cat_idx is None:\n            return x\n        return x[:, self.cat_idx]\n\n    def _x_num(self, x: np.ndarray) -> np.ndarray:\n        return x[:, :0] if self.drop_numeric else x\n\n    def _x_bin(self, x: np.ndarray) -> np.ndarray:\n        return np.zeros_like(x) if self.zero_binary else x\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        row = self.rows[index]\n        shard = self._load_shard(row["shard_file"])\n        i = row["shard_sample_index"]\n        offsets = shard["sample_offsets"]\n        start = int(offsets[i])\n        end = int(offsets[i + 1])\n\n        if end - start != row["event_count"]:\n            raise RuntimeError(\n                f"Manifest/shard event_count mismatch for sample {row[\'sample_id\']}: "\n                f"manifest={row[\'event_count\']}, shard={end - start}"\n            )\n\n        return {\n            "sample_id": row["sample_id"],\n            "label": int(shard["sample_labels"][i]),\n            "event_count": end - start,\n            "x_cat": self._x_cat(shard["x_cat"][start:end]),\n            "x_num": self._x_num(shard["x_num"][start:end]),\n            "x_num_missing": self._x_num(shard["x_num_missing"][start:end]),\n            "x_bin": self._x_bin(shard["x_bin"][start:end]),\n            "delta_t_log1p": shard["delta_t_log1p"][start:end],\n        }\n\n\nclass ReplayBufferDataset(Dataset):\n    """Memory-efficient Dataset reading prototypical OpTC sequences from optc_replay_buffer.npz for Experience Replay."""\n\n    def __init__(self, replay_buffer_path: str | Path) -> None:\n        self.path = Path(replay_buffer_path)\n        if not self.path.exists():\n            raise FileNotFoundError(f"Replay buffer not found at {self.path}")\n\n        z = np.load(self.path, allow_pickle=False)\n        self.sample_offsets = z["sample_offsets"]\n        self.sample_labels = z["sample_labels"]\n        self.x_cat = z["x_cat"]\n        self.x_num = z["x_num"]\n        self.x_num_missing = z["x_num_missing"]\n        self.x_bin = z["x_bin"]\n        self.delta_t_log1p = z["delta_t_log1p"]\n        self._len = len(self.sample_labels)\n        self.pos_indices = np.where(self.sample_labels == 1)[0]\n        self.neg_indices = np.where(self.sample_labels == 0)[0]\n\n    def __len__(self) -> int:\n        return self._len\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        start = int(self.sample_offsets[index])\n        end = int(self.sample_offsets[index + 1])\n        return {\n            "sample_id": f"optc_replay_{index:05d}",\n            "label": int(self.sample_labels[index]),\n            "event_count": end - start,\n            "x_cat": self.x_cat[start:end],\n            "x_num": self.x_num[start:end],\n            "x_num_missing": self.x_num_missing[start:end],\n            "x_bin": self.x_bin[start:end],\n            "delta_t_log1p": self.delta_t_log1p[start:end],\n        }\n\n    def sample_batch(self, n_samples: int, pos_ratio: float = 0.30, max_pos: int = 3) -> list[dict[str, Any]]:\n        """Stratified sample with bounded positive ratio to prevent few-shot domain gradient dominance (MalFSCIL 2024)."""\n        if n_samples <= 0:\n            return []\n        n_pos = min(max_pos, int(round(n_samples * pos_ratio)))\n        if n_pos == 0 and np.random.rand() < pos_ratio:\n            n_pos = 1\n        n_neg = max(0, n_samples - n_pos)\n\n        chosen_indices = []\n        if len(self.pos_indices) > 0 and n_pos > 0:\n            replace_pos = n_pos > len(self.pos_indices)\n            chosen_pos = np.random.choice(self.pos_indices, size=n_pos, replace=replace_pos)\n            chosen_indices.extend(chosen_pos.tolist())\n        if len(self.neg_indices) > 0 and n_neg > 0:\n            replace_neg = n_neg > len(self.neg_indices)\n            chosen_neg = np.random.choice(self.neg_indices, size=n_neg, replace=replace_neg)\n            chosen_indices.extend(chosen_neg.tolist())\n\n        shortfall = n_samples - len(chosen_indices)\n        if shortfall > 0:\n            extra = np.random.choice(self._len, size=shortfall, replace=True)\n            chosen_indices.extend(extra.tolist())\n\n        return [self[idx] for idx in chosen_indices]\n\n\nclass ShardAwareBatchSampler(Sampler[list[int]]):\n    """Batch sampler that batches samples within the same shard to avoid thrashing I/O."""\n\n    def __init__(\n        self,\n        dataset: PackedNetworkDataset,\n        batch_size: int,\n        *,\n        shuffle: bool = True,\n        seed: int = 42,\n        drop_last: bool = False,\n    ) -> None:\n        self.dataset = dataset\n        self.batch_size = int(batch_size)\n        self.shuffle = bool(shuffle)\n        self.seed = int(seed)\n        self.drop_last = bool(drop_last)\n        self.epoch = 0\n\n        if self.batch_size <= 0:\n            raise ValueError(f"batch_size must be > 0, got {self.batch_size}")\n\n        grouped: dict[str, list[int]] = defaultdict(list)\n        for idx, row in enumerate(dataset.rows):\n            grouped[row["shard_file"]].append(idx)\n        self.groups = list(grouped.items())\n\n        total = 0\n        for _, indices in self.groups:\n            if self.drop_last:\n                total += len(indices) // self.batch_size\n            else:\n                total += (len(indices) + self.batch_size - 1) // self.batch_size\n        self._len = total\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __len__(self) -> int:\n        return self._len\n\n    def __iter__(self) -> Iterator[list[int]]:\n        rng = random.Random(self.seed + self.epoch)\n        groups = [(shard, list(indices)) for shard, indices in self.groups]\n        if self.shuffle:\n            rng.shuffle(groups)\n\n        for _, indices in groups:\n            if self.shuffle:\n                rng.shuffle(indices)\n            for start in range(0, len(indices), self.batch_size):\n                batch = indices[start : start + self.batch_size]\n                if self.drop_last and len(batch) < self.batch_size:\n                    continue\n                yield batch\n\n\ndef collate_sequence_batch(batch: list[dict[str, Any]]) -> dict[str, Any]:\n    """Collates a variable-length sequence batch into padded tensors."""\n    if not batch:\n        raise ValueError("Cannot collate empty batch")\n\n    lengths = torch.tensor([x["event_count"] for x in batch], dtype=torch.long)\n    max_len = int(lengths.max().item())\n    bsz = len(batch)\n    c = batch[0]["x_cat"].shape[1]\n    n = batch[0]["x_num"].shape[1]\n    binary_dim = batch[0]["x_bin"].shape[1]\n\n    x_cat = torch.zeros(bsz, max_len, c, dtype=torch.long)\n    x_num = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_missing = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_bin = torch.zeros(bsz, max_len, binary_dim, dtype=torch.float32)\n    delta = torch.zeros(bsz, max_len, 1, dtype=torch.float32)\n    mask = torch.zeros(bsz, max_len, dtype=torch.bool)\n    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)\n\n    for i, item in enumerate(batch):\n        t = item["event_count"]\n        if t > 0:\n            x_cat[i, :t] = torch.from_numpy(np.asarray(item["x_cat"], dtype=np.int64))\n            x_num[i, :t] = torch.from_numpy(np.asarray(item["x_num"], dtype=np.float32))\n            x_missing[i, :t] = torch.from_numpy(np.asarray(item["x_num_missing"], dtype=np.float32))\n            x_bin[i, :t] = torch.from_numpy(np.asarray(item["x_bin"], dtype=np.float32))\n            delta[i, :t, 0] = torch.from_numpy(np.asarray(item["delta_t_log1p"], dtype=np.float32))\n            mask[i, :t] = True\n\n    return {\n        "x_cat": x_cat,\n        "x_num": x_num,\n        "x_num_missing": x_missing,\n        "x_bin": x_bin,\n        "delta_t_log1p": delta,\n        "attention_mask": mask,\n        "labels": labels,\n        "lengths": lengths,\n        "sample_ids": [x["sample_id"] for x in batch],\n    }\n', encoding='utf-8')
(WORK_CODE / 'train_incremental.py').write_text('#!/usr/bin/env python3\n"""Domain-Incremental Learning (Continual Learning) for BiLSTM on ATLASv2.\n\nImplements:\n1. Zero-shot Evaluation of OpTC pretrained model on ATLASv2.\n2. ATLASv2 Scratch Baseline (random initialization).\n3. Continual Learning (IL) from OpTC pretrained model with Experience Replay (80/20 mix).\n4. Backward Transfer (BWT) Evaluation on OpTC test benchmark.\n5. 3-Panel Confusion Matrix generation (comparison_confusion_matrix.png).\n6. Traceback Deep-Dive Analysis for False Negatives & False Positives.\n7. Publication of comparison_metrics.csv matching Host ML Run 6 standards.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef get_autocast(device: torch.device, enabled: bool):\n    if device.type == "cuda":\n        if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):\n            return torch.amp.autocast(device_type="cuda", enabled=enabled)\n        return torch.cuda.amp.autocast(enabled=enabled)\n    return nullcontext()\n\n\ndef get_scaler(device: torch.device, enabled: bool):\n    if device.type == "cuda" and enabled:\n        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):\n            return torch.amp.GradScaler(device="cuda", enabled=True)\n        return torch.cuda.amp.GradScaler(enabled=True)\n\n    class _DummyScaler:\n        def scale(self, loss):\n            return loss\n\n        def step(self, optimizer):\n            optimizer.step()\n\n        def update(self):\n            pass\n\n        def unscale_(self, optimizer):\n            pass\n\n    return _DummyScaler()\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom torch.utils.data import DataLoader\n\ntry:\n    from .bilstm import BiLSTMBaseline\n    from .dataset import (\n        PackedNetworkDataset,\n        ReplayBufferDataset,\n        ShardAwareBatchSampler,\n        collate_sequence_batch,\n    )\nexcept ImportError:\n    from bilstm import BiLSTMBaseline\n    from dataset import (\n        PackedNetworkDataset,\n        ReplayBufferDataset,\n        ShardAwareBatchSampler,\n        collate_sequence_batch,\n    )\n\nSCRIPT_VERSION = "atlasv2-bilstm-incremental-v5.0"\n\n\ndef load_json(path: Path) -> Any:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef dump_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef categorical_cardinalities(dataset_root: Path) -> tuple[list[str], list[int]]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    maps = load_json(dataset_root / "preprocessing" / "network_category_maps.json")\n    fields = list(spec["active_categorical_features"])\n    cards = []\n    for field in fields:\n        mapping = maps["fields"][field]["mapping"]\n        cards.append(max(int(v) for v in mapping.values()) + 1)\n    return fields, cards\n\n\ndef numeric_fields(dataset_root: Path) -> list[str]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    return list(spec.get("active_numerical_features", []))\n\n\ndef make_loader(dataset, *, batch_size: int, shuffle: bool, seed: int, num_workers: int = 0, pin_memory: bool = False):\n    sampler = ShardAwareBatchSampler(\n        dataset,\n        batch_size,\n        shuffle=shuffle,\n        seed=seed,\n        drop_last=False,\n    )\n    loader = DataLoader(\n        dataset,\n        batch_sampler=sampler,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        collate_fn=collate_sequence_batch,\n    )\n    return loader, sampler\n\n\ndef binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, Any]:\n    y_true = np.asarray(y_true, dtype=np.int32)\n    y_prob = np.asarray(y_prob, dtype=np.float64)\n    y_pred = (y_prob >= float(threshold)).astype(np.int32)\n\n    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])\n    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])\n    denom_fpr = fp + tn\n    fpr = float(fp / denom_fpr) if denom_fpr > 0 else 0.0\n\n    try:\n        pr_auc = float(average_precision_score(y_true, y_prob))\n    except Exception:\n        pr_auc = float("nan")\n\n    try:\n        roc_auc = float(roc_auc_score(y_true, y_prob))\n    except Exception:\n        roc_auc = float("nan")\n\n    f1 = float(f1_score(y_true, y_pred, zero_division=0))\n    prec = float(precision_score(y_true, y_pred, zero_division=0))\n    rec = float(recall_score(y_true, y_pred, zero_division=0))\n\n    return {\n        "threshold": float(threshold),\n        "precision": prec,\n        "recall": rec,\n        "f1": f1,\n        "fpr": fpr,\n        "tn": tn,\n        "fp": fp,\n        "fn": fn,\n        "tp": tp,\n        "pr_auc": pr_auc,\n        "roc_auc": roc_auc,\n    }\n\n\ndef make_smoothed_class_weights(y: np.ndarray, device: torch.device, pos_multiplier: float = 2.0) -> torch.Tensor:\n    """Calculate square-root smoothed class weights (Task 3.md Section 1).\n\n    Prevents over-magnification of positive loss when combined with controlled sampling\n    (sample_batch) and Experience Replay.\n    For 1470 neg / 49 pos: sqrt(30) * 2.0 ≈ 10.95 -> clamped into [2.5, 6.0].\n    """\n    n_neg = float(np.sum(y == 0))\n    n_pos = max(float(np.sum(y == 1)), 1.0)\n    raw_ratio = np.sqrt(n_neg / n_pos)\n    w0 = 1.0\n    w1 = min(max(raw_ratio * pos_multiplier, 2.5), 6.0)\n    return torch.tensor([w0, w1], dtype=torch.float32, device=device)\n\n\nclass BinaryFocalLossWithWeights(nn.Module):\n    """Focal Loss with exterior class weights for binary classification (Task 3.md Section 1).\n\n    Computes unweighted BCE to preserve exact pt probability, modulates by (1 - pt)^gamma,\n    and applies smoothed alpha_t externally to avoid MORPH-IDS double-weighting and pt distortion.\n    """\n\n    def __init__(self, alpha: torch.Tensor | None = None, gamma: float = 2.0):\n        super().__init__()\n        self.alpha = alpha  # Tensor [w0, w1] from make_smoothed_class_weights\n        self.gamma = float(gamma)\n\n    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        targets = targets.float().view_as(logits)\n        # 1. Compute unweighted BCE to preserve exact probability pt\n        bce_raw = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")\n        pt = torch.exp(-bce_raw)  # pt = p when y=1, and (1 - p) when y=0\n\n        # 2. Modulate with focal factor (1 - pt)^gamma\n        focal_term = (1.0 - pt) ** self.gamma\n\n        # 3. Apply class weights alpha_t externally\n        if self.alpha is not None:\n            alpha_t = targets * self.alpha[1] + (1.0 - targets) * self.alpha[0]\n            loss = alpha_t * focal_term * bce_raw\n        else:\n            loss = focal_term * bce_raw\n\n        return loss.mean()\n\n\nclass AsymmetricFocalLoss(nn.Module):\n    """Asymmetric Focal Loss (ASL) for hard-negative suppression and extreme class imbalance."""\n\n    def __init__(self, gamma_neg: float = 4.0, gamma_pos: float = 1.0, clip: float = 0.05, eps: float = 1e-8):\n        super().__init__()\n        self.gamma_neg = float(gamma_neg)\n        self.gamma_pos = float(gamma_pos)\n        self.clip = float(clip)\n        self.eps = float(eps)\n\n    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        probs = torch.sigmoid(logits)\n        probs_neg = probs\n        if self.clip > 0:\n            probs_neg = (probs_neg - self.clip).clamp(min=0.0)\n        loss_neg = (1.0 - targets) * torch.pow(probs_neg, self.gamma_neg) * torch.log((1.0 - probs).clamp(min=self.eps))\n        loss_pos = targets * torch.pow(1.0 - probs, self.gamma_pos) * torch.log(probs.clamp(min=self.eps))\n        return -(loss_pos + loss_neg).mean()\n\n\nclass SupConWithHardNegativeRepulsion(nn.Module):\n    """Supervised Contrastive Loss with Hard Negative Repulsion (Run 8 / Task 5.md Thay đổi 2).\n\n    Combines standard SupCon (Khosla et al., 2020) with active cosine repulsion for hard negatives:\n    penalizes negative samples whose cosine similarity with any positive sample exceeds margin (default 0.15).\n    This actively forces benign background traffic (e.g. firefox.exe HTTPS windows and VM startup footprints)\n    away from the attack cluster into the low-probability zone (< 0.25).\n    """\n\n    def __init__(self, temperature: float = 0.1, margin: float = 0.15, repulsion_weight: float = 0.5):\n        super().__init__()\n        self.temperature = float(temperature)\n        self.margin = float(margin)\n        self.repulsion_weight = float(repulsion_weight)\n\n    def forward(self, features: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        device = features.device\n        batch_size = features.shape[0]\n        if batch_size <= 1:\n            return torch.tensor(0.0, device=device)\n\n        targets = targets.view(-1, 1)\n        # mask[i, j] = 1 if sample i and sample j share the same label\n        mask = torch.eq(targets, targets.T).float().to(device)\n\n        # L2-normalize feature representations\n        features_norm = F.normalize(features, p=2, dim=1)\n        cos_sim = torch.matmul(features_norm, features_norm.T)\n        sim_matrix = cos_sim / self.temperature\n\n        # Subtract max for numerical stability\n        sim_max, _ = torch.max(sim_matrix, dim=1, keepdim=True)\n        sim_matrix = sim_matrix - sim_max.detach()\n\n        # Mask out self-contrast (diagonal entries)\n        logits_mask = torch.scatter(\n            torch.ones_like(mask),\n            1,\n            torch.arange(batch_size, device=device).view(-1, 1),\n            0.0,\n        )\n        pos_mask = mask * logits_mask\n\n        exp_sim = torch.exp(sim_matrix) * logits_mask\n        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)\n\n        denom = pos_mask.sum(dim=1).clamp(min=1e-8)\n        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / denom\n\n        valid_anchors = (pos_mask.sum(dim=1) > 0)\n        supcon_loss = -mean_log_prob_pos[valid_anchors].mean() if valid_anchors.any() else torch.tensor(0.0, device=device)\n\n        # Hard Negative Repulsion:\n        # For all pairs where target_i == 1 (positive) and target_j == 0 (negative),\n        # penalize any cosine similarity that exceeds margin (0.15)\n        is_pos = (targets == 1).view(-1, 1)\n        is_neg = (targets == 0).view(1, -1)\n        pos_neg_mask = (is_pos & is_neg)\n        if pos_neg_mask.any():\n            pos_neg_sim = cos_sim[pos_neg_mask]\n            repulsion_loss = F.relu(pos_neg_sim - self.margin).pow(2).mean()\n        else:\n            repulsion_loss = torch.tensor(0.0, device=device)\n\n        return supcon_loss + self.repulsion_weight * repulsion_loss\n\n\nSupervisedContrastiveLoss = SupConWithHardNegativeRepulsion  # Alias for backward compatibility\n\n\ndef find_optimal_threshold(\n    y_val_true: np.ndarray,\n    y_val_probs: np.ndarray,\n    *,\n    t_min: float = 0.20,\n    t_max: float = 0.50,\n    beta: float = 1.0,\n    soft_max_fpr: float = 0.08,\n    num_points: int = 71,\n) -> tuple[float, dict[str, Any]]:\n    """Controlled threshold sweep in [t_min, t_max] with balanced F_beta (default beta=1.0) and soft FPR penalty.\n\n    - Clamps lower bound at t_min = 0.20 to eliminate background noise causing false alarms.\n    - Default beta = 1.0 seeks the true harmonic mean between Precision and Recall.\n    - Soft exponential penalty if validation FPR exceeds soft_max_fpr (8%).\n    - Moving average smoothing (kernel=3) for plateau threshold stability.\n    """\n    y_val_true = np.asarray(y_val_true, dtype=np.int32)\n    y_val_probs = np.asarray(y_val_probs, dtype=np.float64)\n\n    thresholds = np.linspace(t_min, t_max, num_points)\n    scores = []\n    n_neg = max(int(np.sum(y_val_true == 0)), 1)\n\n    for t in thresholds:\n        preds = (y_val_probs >= t).astype(np.int32)\n        tp = int(np.sum((preds == 1) & (y_val_true == 1)))\n        fp = int(np.sum((preds == 1) & (y_val_true == 0)))\n        fn = int(np.sum((preds == 0) & (y_val_true == 1)))\n\n        if tp == 0:\n            scores.append(0.0)\n            continue\n\n        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n        denom = (beta**2 * prec) + rec + 1e-8\n        f_beta = ((1.0 + beta**2) * (prec * rec)) / denom\n\n        fpr = fp / n_neg\n        if fpr > soft_max_fpr:\n            f_beta *= float(np.exp(-5.0 * (fpr - soft_max_fpr)))\n\n        scores.append(float(f_beta))\n\n    scores = np.array(scores, dtype=np.float64)\n\n    if np.max(scores) <= 0.0:\n        best_t = float((t_min + t_max) / 2.0)\n        return best_t, binary_metrics(y_val_true, y_val_probs, best_t)\n\n    smoothed_scores = np.convolve(scores, np.ones(3) / 3.0, mode="same")\n    best_idx = int(np.argmax(smoothed_scores))\n    best_t = float(thresholds[best_idx])\n    return best_t, binary_metrics(y_val_true, y_val_probs, best_t)\n\n\ndef find_optimal_threshold_run6(y_val_true, y_val_probs, **kwargs):\n    """Alias pointing to find_optimal_threshold."""\n    return find_optimal_threshold(y_val_true, y_val_probs, **kwargs)\n\n\ndef find_best_f1_threshold(y_true, y_prob, **kwargs):\n    """Legacy alias routing to controlled threshold search for backward compatibility."""\n    return find_optimal_threshold(y_true, y_prob, t_min=0.20, t_max=0.50)\n\n\ndef find_harmonic_domain_threshold(\n    y_atlas_true: np.ndarray,\n    y_atlas_probs: np.ndarray,\n    y_optc_true: np.ndarray | None = None,\n    y_optc_probs: np.ndarray | None = None,\n    *,\n    t_min: float = 0.25,\n    t_max: float = 0.75,\n    num_points: int = 101,\n) -> tuple[float, dict[str, Any]]:\n    """Harmonic Domain-Balanced Validation Threshold Search (Run 8 / Task 5.md Thay đổi 4).\n\n    Maximizes the Harmonic Mean of F1 between ATLASv2 and OpTC domains across candidate thresholds:\n        H_F1(t) = 2 * F1_atlas(t) * F1_optc(t) / (F1_atlas(t) + F1_optc(t) + 1e-8)\n    Prevents OpTC positive dominance while guaranteeing an optimal operating point for both domains.\n    """\n    thresholds = np.linspace(t_min, t_max, num_points)\n    scores = []\n\n    for t in thresholds:\n        m_atlas = binary_metrics(y_atlas_true, y_atlas_probs, t)\n        f1_atlas = m_atlas["f1"]\n\n        if y_optc_true is not None and y_optc_probs is not None:\n            m_optc = binary_metrics(y_optc_true, y_optc_probs, t)\n            f1_optc = m_optc["f1"]\n            denom = f1_atlas + f1_optc + 1e-8\n            harmonic_f1 = (2.0 * f1_atlas * f1_optc) / denom\n            scores.append(float(harmonic_f1))\n        else:\n            scores.append(float(f1_atlas))\n\n    scores = np.array(scores, dtype=np.float64)\n    if np.max(scores) <= 0.0:\n        best_t = float((t_min + t_max) / 2.0)\n    else:\n        smoothed = np.convolve(scores, np.ones(3) / 3.0, mode="same")\n        best_idx = int(np.argmax(smoothed))\n        best_t = float(thresholds[best_idx])\n\n    if y_optc_true is not None and y_optc_probs is not None:\n        y_comb_true = np.concatenate([y_atlas_true, y_optc_true])\n        y_comb_probs = np.concatenate([y_atlas_probs, y_optc_probs])\n        combined_m = binary_metrics(y_comb_true, y_comb_probs, best_t)\n        m_atlas = binary_metrics(y_atlas_true, y_atlas_probs, best_t)\n        m_optc = binary_metrics(y_optc_true, y_optc_probs, best_t)\n        combined_m["f1_atlas"] = m_atlas["f1"]\n        combined_m["f1_optc"] = m_optc["f1"]\n        denom = m_atlas["f1"] + m_optc["f1"] + 1e-8\n        combined_m["harmonic_f1"] = (2.0 * m_atlas["f1"] * m_optc["f1"]) / denom\n        return best_t, combined_m\n    else:\n        return best_t, binary_metrics(y_atlas_true, y_atlas_probs, best_t)\n\n\ndef evaluate_domain_validation(\n    model: nn.Module,\n    val_loader: DataLoader,\n    replay_ds: ReplayBufferDataset | None,\n    device: torch.device,\n    amp: bool,\n    max_replay_val: int = 150,\n) -> tuple[np.ndarray, np.ndarray, list[str], np.ndarray | None, np.ndarray | None, list[str] | None]:\n    """Action 1 & Task 5.md Thay đổi 4: Evaluates ATLASv2 validation and OpTC replay buffer separately."""\n    atlas_true, atlas_prob, atlas_ids = predict(model, val_loader, device=device, amp=amp)\n    rep_labels, rep_probs, rep_ids = None, None, None\n    if replay_ds is not None and len(replay_ds) > 0:\n        n_rep = min(max_replay_val, len(replay_ds))\n        rep_indices = np.linspace(0, len(replay_ds) - 1, n_rep, dtype=int)\n        rep_items = [replay_ds[i] for i in rep_indices]\n        rep_batch = collate_sequence_batch(rep_items)\n        model.eval()\n        with torch.no_grad():\n            with get_autocast(device, amp):\n                rep_logits = model(\n                    x_cat=rep_batch["x_cat"].to(device),\n                    x_num=rep_batch["x_num"].to(device),\n                    x_num_missing=rep_batch["x_num_missing"].to(device),\n                    x_bin=rep_batch["x_bin"].to(device),\n                    delta_t_log1p=rep_batch["delta_t_log1p"].to(device),\n                    lengths=rep_batch["lengths"].to(device),\n                )\n                rep_probs = torch.sigmoid(rep_logits).cpu().numpy()\n        rep_labels = rep_batch["labels"].numpy()\n        rep_ids = rep_batch["sample_ids"]\n    return atlas_true, atlas_prob, atlas_ids, rep_labels, rep_probs, rep_ids\n\n\ndef evaluate_combined_validation(\n    model: nn.Module,\n    val_loader: DataLoader,\n    replay_ds: ReplayBufferDataset | None,\n    device: torch.device,\n    amp: bool,\n    max_replay_val: int = 150,\n) -> tuple[np.ndarray, np.ndarray, list[str]]:\n    """Legacy helper combining both domains into a single flat array."""\n    atlas_true, atlas_prob, atlas_ids, rep_labels, rep_probs, rep_ids = evaluate_domain_validation(\n        model, val_loader, replay_ds, device, amp, max_replay_val\n    )\n    if rep_labels is not None and rep_probs is not None:\n        val_true = np.concatenate([atlas_true, rep_labels])\n        val_prob = np.concatenate([atlas_prob, rep_probs])\n        val_ids = atlas_ids + (rep_ids or [])\n        return val_true, val_prob, val_ids\n    return atlas_true, atlas_prob, atlas_ids\n\n\n\ndef load_pretrained_checkpoint(model: nn.Module, ckpt_path: str | Path, device: torch.device) -> dict[int, int]:\n    """Load pretrained model weights, adapting embedding layers when vocabulary is expanded."""\n    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)\n    state_dict = ckpt.get("model_state_dict", ckpt)\n    model_state = model.state_dict()\n    old_vocab_sizes: dict[int, int] = {}\n\n    for key, val in state_dict.items():\n        if key not in model_state:\n            continue\n        if model_state[key].shape == val.shape:\n            model_state[key] = val\n        elif "embeddings" in key and model_state[key].dim() == 2 and val.dim() == 2:\n            try:\n                emb_idx = int(key.split(".")[1])\n            except Exception:\n                emb_idx = -1\n            n_old = min(val.shape[0], model_state[key].shape[0])\n            d = min(val.shape[1], model_state[key].shape[1])\n            model_state[key][:n_old, :d] = val[:n_old, :d]\n            if emb_idx >= 0:\n                old_vocab_sizes[emb_idx] = n_old\n            if model_state[key].shape[0] > n_old:\n                unk_vec = val[0:1, :d]  # row 0 is <UNK>\n                model_state[key][n_old:, :d] = unk_vec\n                print(f"  [Vocab Expansion] {key}: transferred {n_old} rows from OpTC, initialized {model_state[key].shape[0] - n_old} new rows with <UNK> embedding.")\n        else:\n            print(f"  [Warning] Skipping key {key} due to shape mismatch: ckpt {val.shape} vs model {model_state[key].shape}")\n\n    model.load_state_dict(model_state)\n    print(f"Loaded pretrained weights from {ckpt_path}")\n    return old_vocab_sizes\n\n\ndef apply_optc_gradient_shield(model: nn.Module, old_sizes: dict[int, int], old_scale: float = 0.35) -> None:\n    """Shield shared OpTC vocabulary from rapid drift during incremental training (Run 8 / Task 5.md Thay đổi 3)."""\n    if not old_sizes:\n        return\n    for i, emb in enumerate(model.embeddings):\n        n_old = old_sizes.get(i, 0)\n        if n_old > 0 and emb.weight.shape[0] > n_old:\n            def _make_hook(cutoff: int, scale: float):\n                def _hook(grad: torch.Tensor) -> torch.Tensor:\n                    g = grad.clone()\n                    g[:cutoff] *= scale\n                    return g\n                return _hook\n            emb.weight.register_hook(_make_hook(n_old, old_scale))\n            print(f"  [Gradient Shield] Protected {n_old} OpTC rows in embeddings[{i}] (grad_scale={old_scale}x); {emb.weight.shape[0] - n_old} new ATLASv2 rows train at full speed (1.0x).")\n\n\n\ndef predict(model: nn.Module, loader: DataLoader, device: torch.device, amp: bool = False) -> tuple[np.ndarray, np.ndarray, list[str]]:\n    model.eval()\n    all_prob = []\n    all_true = []\n    all_ids = []\n\n    with torch.no_grad():\n        for batch in loader:\n            x_cat = batch["x_cat"].to(device)\n            x_num = batch["x_num"].to(device)\n            x_num_missing = batch["x_num_missing"].to(device)\n            x_bin = batch["x_bin"].to(device)\n            delta = batch["delta_t_log1p"].to(device)\n            mask = batch["attention_mask"].to(device)\n            lengths = batch["lengths"].to(device)\n\n            with get_autocast(device, amp):\n                logits = model(\n                    x_cat=x_cat,\n                    x_num=x_num,\n                    x_num_missing=x_num_missing,\n                    x_bin=x_bin,\n                    delta_t_log1p=delta,\n                    lengths=lengths,\n                )\n                probs = torch.sigmoid(logits)\n\n            all_prob.extend(probs.detach().cpu().numpy().tolist())\n            all_true.extend(batch["labels"].numpy().tolist())\n            all_ids.extend(batch["sample_ids"])\n\n    return np.asarray(all_true, dtype=np.int32), np.asarray(all_prob, dtype=np.float64), all_ids\n\n\ndef train_single_model(\n    model: nn.Module,\n    train_ds: PackedNetworkDataset,\n    val_loader: DataLoader,\n    test_loader: DataLoader,\n    *,\n    device: torch.device,\n    output_dir: Path,\n    epochs: int = 10,\n    min_epochs: int = 3,\n    batch_size: int = 32,\n    lr: float = 2e-4,\n    weight_decay: float = 1e-4,\n    grad_clip: float = 1.0,\n    patience: int = 4,\n    amp: bool = False,\n    seed: int = 42,\n    replay_ds: ReplayBufferDataset | None = None,\n    replay_ratio: float = 0.05,\n    model_tag: str = "model",\n    freeze_epochs: int = 0,\n    loss_type: str = "focal",\n    t_min: float = 0.20,\n    t_max: float = 0.55,\n) -> dict[str, Any]:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    best_path = output_dir / f"{model_tag}_best.pt"\n\n    train_labels = np.array([row["label"] for row in train_ds.rows], dtype=np.int32)\n    train_pos = int(np.sum(train_labels == 1))\n    train_neg = int(np.sum(train_labels == 0))\n\n    # Loss Criterion Setup (Task 3.md Section 1: BinaryFocalLossWithWeights with smoothed weights)\n    if loss_type == "focal":\n        alpha = make_smoothed_class_weights(train_labels, device=device, pos_multiplier=2.0)\n        criterion = BinaryFocalLossWithWeights(alpha=alpha, gamma=2.0)\n        print(f"  [Loss] Configured BinaryFocalLossWithWeights (gamma=2.0, alpha=[{alpha[0]:.2f}, {alpha[1]:.2f}])")\n    elif loss_type == "asl":\n        criterion = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05)\n        print("  [Loss] Configured Asymmetric Focal Loss (ASL: gamma_neg=4.0, gamma_pos=1.0, clip=0.05)")\n    else:\n        raw_ratio = float(train_neg / max(1, train_pos))\n        clamped_pos_weight = min(10.0, max(2.0, raw_ratio * 0.25))\n        pos_weight = torch.tensor([clamped_pos_weight], device=device)\n        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n        print(f"  [Loss] Configured BCEWithLogitsLoss (pos_weight={clamped_pos_weight:.2f})")\n\n    # Supervised Contrastive Loss with Hard Negative Repulsion (Run 8 / Task 5.md Thay đổi 2)\n    contrastive_criterion = SupConWithHardNegativeRepulsion(\n        temperature=0.1, margin=0.15, repulsion_weight=0.5\n    ) if (model_tag == "incremental") else None\n    if contrastive_criterion is not None:\n        print("  [Loss] Configured SupConWithHardNegativeRepulsion (tau=0.1, margin=0.15, repulsion_wt=0.5, total_wt=0.25).")\n\n    # Mở khóa toàn bộ tham số mô hình (Task 3.md Section 2)\n    for param in model.parameters():\n        param.requires_grad = True\n\n    # 3-Tier Decoupled Learning Rate Setup (Task 3.md Section 2)\n    if model_tag == "incremental":\n        lr_emb = lr * 2.5       # 5e-4 when lr=2e-4: allows 80 new ATLASv2 binaries to escape <UNK>\n        lr_backbone = lr * 0.5  # 1e-4 when lr=2e-4: preserves temporal sequence patterns from OpTC\n        lr_head = lr * 5.0      # 1e-3 when lr=2e-4: rapidly shifts logit scale from +5.4 to balanced range\n\n        backbone_params = list(model.event_encoder.parameters()) + list(model.lstm.parameters())\n        optimizer = torch.optim.AdamW([\n            {"params": model.embeddings.parameters(), "lr": lr_emb, "weight_decay": weight_decay},\n            {"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay},\n            {"params": model.classifier.parameters(), "lr": lr_head, "weight_decay": weight_decay},\n        ])\n        print(f"  [3-Tier Decoupled LR] Embeddings: {lr_emb:.2e} | Backbone (Encoder+LSTM): {lr_backbone:.2e} | Classifier Head: {lr_head:.2e}")\n    else:\n        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)\n\n    scaler = get_scaler(device, amp)\n\n    best_val_score = -1.0\n    best_epoch = 0\n    stale_epochs = 0\n    history = []\n\n    print(f"\\n--- Training {model_tag.upper()} ({epochs} epochs, lr={lr}, ER replay={replay_ds is not None}) ---")\n\n    for epoch in range(1, epochs + 1):\n        t_start = time.time()\n        model.train()\n        train_loader, sampler = make_loader(train_ds, batch_size=batch_size, shuffle=True, seed=seed + epoch)\n        sampler.set_epoch(epoch)\n\n        running_loss = 0.0\n        seen_batches = 0\n\n        for raw_batch_items in sampler:\n            batch_items = [train_ds[i] for i in raw_batch_items]\n\n            # Interleave Experience Replay buffer samples if available (Adaptive Replay Scheduling - MalFSCIL 2024)\n            if replay_ds is not None and len(replay_ds) > 0 and replay_ratio > 0:\n                n_replay = max(1, min(2, int(round(len(batch_items) * replay_ratio))))\n                if hasattr(replay_ds, "sample_batch"):\n                    batch_items.extend(replay_ds.sample_batch(n_replay, pos_ratio=0.30, max_pos=3))\n                else:\n                    replay_indices = np.random.choice(len(replay_ds), size=n_replay, replace=False)\n                    for repl_idx in replay_indices:\n                        batch_items.append(replay_ds[repl_idx])\n\n            batch = collate_sequence_batch(batch_items)\n\n            x_cat = batch["x_cat"].to(device)\n            x_num = batch["x_num"].to(device)\n            x_num_missing = batch["x_num_missing"].to(device)\n            x_bin = batch["x_bin"].to(device)\n            delta = batch["delta_t_log1p"].to(device)\n            lengths = batch["lengths"].to(device)\n            targets = batch["labels"].to(device)\n\n            optimizer.zero_grad(set_to_none=True)\n            with get_autocast(device, amp):\n                if contrastive_criterion is not None:\n                    logits, pooled = model(\n                        x_cat=x_cat,\n                        x_num=x_num,\n                        x_num_missing=x_num_missing,\n                        x_bin=x_bin,\n                        delta_t_log1p=delta,\n                        lengths=lengths,\n                        return_features=True,\n                    )\n                    loss_focal = criterion(logits, targets)\n                    loss_con = contrastive_criterion(pooled, targets)\n                    loss = loss_focal + 0.25 * loss_con\n                else:\n                    logits = model(\n                        x_cat=x_cat,\n                        x_num=x_num,\n                        x_num_missing=x_num_missing,\n                        x_bin=x_bin,\n                        delta_t_log1p=delta,\n                        lengths=lengths,\n                    )\n                    loss = criterion(logits, targets)\n\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n\n            running_loss += float(loss.item())\n            seen_batches += 1\n\n        train_loss = running_loss / max(1, seen_batches)\n\n        # Harmonic Domain-Balanced Validation (Run 8 / Task 5.md Thay đổi 4)\n        if model_tag == "incremental" and replay_ds is not None:\n            atlas_true, atlas_prob, _, rep_true, rep_prob, _ = evaluate_domain_validation(\n                model, val_loader, replay_ds, device=device, amp=amp\n            )\n            val_thr, val_metrics = find_harmonic_domain_threshold(\n                atlas_true, atlas_prob, rep_true, rep_prob, t_min=t_min, t_max=t_max\n            )\n            val_comb_true = np.concatenate([atlas_true, rep_true])\n            val_comb_prob = np.concatenate([atlas_prob, rep_prob])\n            val_pr_auc = float(average_precision_score(val_comb_true, val_comb_prob))\n            # Checkpoint selection prioritizes Harmonic F1 across both domains\n            val_score = val_metrics.get("harmonic_f1", val_metrics["f1"]) + 0.1 * val_pr_auc\n        else:\n            val_true, val_prob, _ = predict(model, val_loader, device=device, amp=amp)\n            val_pr_auc = float(average_precision_score(val_true, val_prob))\n            val_thr, val_metrics = find_optimal_threshold(val_true, val_prob, t_min=t_min, t_max=t_max, beta=1.0)\n            val_score = val_pr_auc + 0.1 * val_metrics["f1"]\n\n        elapsed = time.time() - t_start\n\n        row = {\n            "epoch": epoch,\n            "train_loss": train_loss,\n            "val_pr_auc": val_pr_auc,\n            "val_f1": val_metrics["f1"],\n            "val_recall": val_metrics["recall"],\n            "val_precision": val_metrics["precision"],\n            "val_fpr": val_metrics["fpr"],\n            "val_threshold": val_thr,\n            "elapsed_seconds": elapsed,\n        }\n        history.append(row)\n        print(f"  [{model_tag} Ep {epoch:2d}/{epochs:2d}] Loss: {train_loss:.4f} | Val PR-AUC: {val_pr_auc:.4f} | Val F1: {val_metrics[\'f1\']:.4f} | Thr: {val_thr:.4f} ({elapsed:.1f}s)")\n\n        # Warm-up Checkpoint & Early Stopping Policy (Task 3.md Section 3)\n        if val_score > best_val_score + 1e-6:\n            best_val_score = val_score\n            best_epoch = epoch\n            stale_epochs = 0\n            torch.save({\n                "model_state_dict": model.state_dict(),\n                "best_epoch": best_epoch,\n                "best_val_pr_auc": val_pr_auc,\n                "best_val_score": best_val_score,\n            }, best_path)\n        else:\n            if epoch >= min_epochs:\n                stale_epochs += 1\n\n        if stale_epochs >= patience:\n            print(f"  [{model_tag} Early Stopping] No improvement in Val score for {stale_epochs} epochs (after epoch {min_epochs}).")\n            break\n\n    # Load best checkpoint\n    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)\n    model.load_state_dict(best_ckpt["model_state_dict"])\n\n    # Optimize threshold on VAL (Harmonic Domain-Balanced Validation for Run 8)\n    if model_tag == "incremental" and replay_ds is not None:\n        atlas_true, atlas_prob, atlas_ids, rep_true, rep_prob, rep_ids = evaluate_domain_validation(\n            model, val_loader, replay_ds, device=device, amp=amp\n        )\n        final_thr, final_val_m = find_harmonic_domain_threshold(\n            atlas_true, atlas_prob, rep_true, rep_prob, t_min=t_min, t_max=t_max\n        )\n    else:\n        val_true, val_prob, val_ids = predict(model, val_loader, device=device, amp=amp)\n        final_thr, final_val_m = find_optimal_threshold(val_true, val_prob, t_min=t_min, t_max=t_max, beta=1.0)\n\n    # Evaluate on TEST (frozen threshold)\n    test_true, test_prob, test_ids = predict(model, test_loader, device=device, amp=amp)\n    final_test_m = binary_metrics(test_true, test_prob, final_thr)\n\n    # Save predictions\n    pred_csv = output_dir / f"{model_tag}_test_predictions.csv"\n    with open(pred_csv, "w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(["sample_id", "label", "probability", "prediction"])\n        for sid, yt, yp in zip(test_ids, test_true, test_prob):\n            writer.writerow([sid, yt, float(yp), int(yp >= final_thr)])\n\n    return {\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_ckpt.get("best_val_pr_auc", final_val_m["pr_auc"]),\n        "threshold": final_thr,\n        "val": final_val_m,\n        "test": final_test_m,\n        "test_true": test_true,\n        "test_prob": test_prob,\n        "test_ids": test_ids,\n        "model": model,\n    }\n\n\ndef plot_3panel_confusion_matrix(\n    zs_metrics: dict[str, Any],\n    scratch_metrics: dict[str, Any],\n    il_metrics: dict[str, Any],\n    output_path: Path,\n) -> None:\n    try:\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n\n        fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n        panels = [\n            ("OpTC Base Model (Zero-Shot)", zs_metrics),\n            ("ATLASv2 Scratch (Baseline)", scratch_metrics),\n            ("OpTC -> ATLASv2 Continual (IL + ER)", il_metrics),\n        ]\n\n        for ax, (title, m) in zip(axes, panels):\n            cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])\n            cax = ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.7)\n            for i in range(2):\n                for j in range(2):\n                    ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14, fontweight="bold")\n            ax.set_xticks([0, 1])\n            ax.set_yticks([0, 1])\n            ax.set_xticklabels(["Pred Neg (0)", "Pred Pos (1)"], fontsize=11)\n            ax.set_yticklabels(["True Neg (0)", "True Pos (1)"], fontsize=11)\n            ax.set_title(f"{title}\\nF1: {m[\'f1\']:.4f} | Prec: {m[\'precision\']:.4f} | Rec: {m[\'recall\']:.4f}", fontsize=12, pad=15)\n\n        fig.tight_layout()\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(output_path, dpi=200, bbox_inches="tight")\n        plt.close(fig)\n        print(f"Saved 3-panel confusion matrix: {output_path}")\n    except Exception as e:\n        print(f"Warning: could not generate confusion matrix plot ({e})")\n\n\ndef run_traceback_analysis(\n    test_ds: PackedNetworkDataset,\n    test_ids: list[str],\n    y_true: np.ndarray,\n    y_prob: np.ndarray,\n    threshold: float,\n    output_path: Path,\n) -> list[dict[str, Any]]:\n    manifest_map = {row["sample_id"]: row for row in test_ds.rows}\n    trace_rows = []\n\n    for sid, yt, yp in zip(test_ids, y_true, y_prob):\n        pred = int(yp >= threshold)\n        if pred != yt:\n            m_row = manifest_map.get(sid, {})\n            err_type = "FN" if yt == 1 else "FP"\n            trace_rows.append({\n                "sample_id": sid,\n                "error_type": err_type,\n                "label": int(yt),\n                "probability": float(yp),\n                "threshold": float(threshold),\n                "event_count": int(m_row.get("event_count", 0)),\n                "shard_file": m_row.get("shard_file", ""),\n                "hostname": m_row.get("hostname", "unknown"),\n                "window_start_utc": m_row.get("window_start_utc", "unknown"),\n            })\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with open(output_path, "w", newline="", encoding="utf-8") as f:\n        fieldnames = ["sample_id", "error_type", "label", "probability", "threshold", "event_count", "shard_file", "hostname", "window_start_utc"]\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(trace_rows)\n\n    print(f"Saved Traceback Analysis: {output_path} ({len(trace_rows)} error samples found: {sum(1 for r in trace_rows if r[\'error_type\'] == \'FN\')} FN, {sum(1 for r in trace_rows if r[\'error_type\'] == \'FP\')} FP)")\n    return trace_rows\n\n\ndef main():\n    p = argparse.ArgumentParser(description="BiLSTM Incremental Learning & Benchmark on ATLASv2")\n    p.add_argument("--dataset-root", required=True, help="Path to preprocessed dataset root")\n    p.add_argument("--output-dir", required=True, help="Directory to save model checkpoints and logs")\n    p.add_argument("--init-model", default=None, help="Path to OpTC pretrained checkpoint (best_model.pt)")\n    p.add_argument("--replay-buffer", default=None, help="Path to OpTC Experience Replay buffer (optc_replay_buffer.npz)")\n    p.add_argument("--replay-ratio", type=float, default=0.05, help="Fraction of replay samples per mini-batch (default 0.05 - MalFSCIL 2024)")\n    p.add_argument("--optc-test-root", default=None, help="Path to optc_test/ directory for Backward Transfer evaluation")\n    p.add_argument("--run-benchmark", action="store_true", help="Run full 4-experiment benchmark suite (Scratch, Zero-shot, IL, BWT)")\n    p.add_argument("--epochs", type=int, default=10, help="Number of training epochs")\n    p.add_argument("--min-epochs", type=int, default=3, help="Minimum warmup epochs before allowing early stopping (default: 3)")\n    p.add_argument("--freeze-epochs", type=int, default=0, help="Legacy parameter, kept for compatibility (default: 0)")\n    p.add_argument("--loss", default="focal", choices=["focal", "asl", "bce"], help="Loss function for training (default: focal - MORPH-IDS BinaryFocalLossWithWeights)")\n    p.add_argument("--t-min", type=float, default=0.25, help="Minimum threshold for controlled search (default: 0.25 - Run 8)")\n    p.add_argument("--t-max", type=float, default=0.75, help="Maximum threshold for controlled search (default: 0.75 - Run 8)")\n    p.add_argument("--max-fpr", type=float, default=0.03, help="Maximum validation FPR constraint (legacy)")\n    p.add_argument("--batch-size", type=int, default=32, help="Mini-batch size")\n    p.add_argument("--lr", type=float, default=2e-4, help="Learning rate for incremental fine-tuning")\n    p.add_argument("--weight-decay", type=float, default=1e-4)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--patience", type=int, default=4, help="Early stopping patience")\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--no-amp", action="store_true")\n    args = p.parse_args()\n\n    dataset_root = Path(args.dataset_root)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    results_dir = output_dir / "results"\n    results_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(args.seed)\n\n    cat_fields, cat_cards = categorical_cardinalities(dataset_root)\n    num_fields = numeric_fields(dataset_root)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    amp = (not args.no_amp) and device.type == "cuda"\n\n    print("=" * 80)\n    print("BILSTM CONTINUAL LEARNING BENCHMARK ON ATLASv2")\n    print("=" * 80)\n    print(f"Dataset Root:      {dataset_root}")\n    print(f"Output Dir:        {output_dir}")\n    print(f"Device:            {device} (AMP: {amp})")\n    print(f"Replay Buffer:     {args.replay_buffer} (ratio: {args.replay_ratio})")\n    print(f"OpTC Test Set:     {args.optc_test_root}")\n\n    train_ds = PackedNetworkDataset(dataset_root, "train")\n    val_ds = PackedNetworkDataset(dataset_root, "val")\n    test_ds = PackedNetworkDataset(dataset_root, "test")\n\n    val_loader, _ = make_loader(val_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed)\n    test_loader, _ = make_loader(test_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed)\n\n    replay_ds = None\n    if args.replay_buffer and Path(args.replay_buffer).exists():\n        replay_ds = ReplayBufferDataset(args.replay_buffer)\n        print(f"Loaded Experience Replay Buffer: {len(replay_ds)} samples")\n\n    # Helper function to create fresh BiLSTM model instance\n    def create_model():\n        return BiLSTMBaseline(\n            categorical_cardinalities=cat_cards,\n            numerical_dim=len(num_fields),\n            event_dim=64,\n            hidden_dim=64,\n            lstm_layers=1,\n            dropout=0.2,\n        ).to(device)\n\n    # -------------------------------------------------------------\n    # 1. Zero-shot Evaluation of OpTC Pretrained Model on ATLASv2\n    # -------------------------------------------------------------\n    print("\\n" + "=" * 60)\n    print("EXPERIMENT 1: OpTC Pretrained Model (Zero-Shot on ATLASv2)")\n    print("=" * 60)\n    zs_model = create_model()\n    if args.init_model and Path(args.init_model).exists():\n        load_pretrained_checkpoint(zs_model, args.init_model, device)\n\n    zs_val_true, zs_val_prob, _ = predict(zs_model, val_loader, device=device, amp=amp)\n    zs_test_true, zs_test_prob, zs_test_ids = predict(zs_model, test_loader, device=device, amp=amp)\n    optc_frozen_thr = 0.9956\n    zs_val_m = binary_metrics(zs_val_true, zs_val_prob, optc_frozen_thr)\n    zs_test_m = binary_metrics(zs_test_true, zs_test_prob, optc_frozen_thr)\n    print(f"  Zero-Shot Test: F1={zs_test_m[\'f1\']:.4f} | Prec={zs_test_m[\'precision\']:.4f} | Rec={zs_test_m[\'recall\']:.4f} | TP={zs_test_m[\'tp\']}, FP={zs_test_m[\'fp\']}, FN={zs_test_m[\'fn\']}")\n\n    # -------------------------------------------------------------\n    # 2. ATLASv2 Scratch Baseline (Random Initialization)\n    # -------------------------------------------------------------\n    print("\\n" + "=" * 60)\n    print("EXPERIMENT 2: ATLASv2 Scratch Baseline (From Scratch)")\n    print("=" * 60)\n    scratch_model = create_model()\n    scratch_res = train_single_model(\n        scratch_model,\n        train_ds,\n        val_loader,\n        test_loader,\n        device=device,\n        output_dir=results_dir / "scratch",\n        epochs=args.epochs,\n        min_epochs=args.min_epochs,\n        batch_size=args.batch_size,\n        lr=args.lr * 2,  # scratch typically trains with higher lr\n        patience=args.patience,\n        amp=amp,\n        seed=args.seed,\n        replay_ds=None,  # No replay for scratch baseline\n        model_tag="scratch",\n        freeze_epochs=0,\n        loss_type="focal",\n        t_min=args.t_min,\n        t_max=args.t_max,\n    )\n    scratch_test_m = scratch_res["test"]\n    scratch_val_m = scratch_res["val"]\n    print(f"  Scratch Baseline Test: F1={scratch_test_m[\'f1\']:.4f} | Prec={scratch_test_m[\'precision\']:.4f} | Rec={scratch_test_m[\'recall\']:.4f} | TP={scratch_test_m[\'tp\']}, FP={scratch_test_m[\'fp\']}, FN={scratch_test_m[\'fn\']}")\n\n    # -------------------------------------------------------------\n    # 3. OpTC -> ATLASv2 Continual Learning (IL with Replay Buffer)\n    # -------------------------------------------------------------\n    print("\\n" + "=" * 60)\n    print("EXPERIMENT 3: OpTC -> ATLASv2 Continual Learning (Domain-IL with ER)")\n    print("=" * 60)\n    il_model = create_model()\n    old_vocab_sizes: dict[int, int] = {}\n    if args.init_model and Path(args.init_model).exists():\n        old_vocab_sizes = load_pretrained_checkpoint(il_model, args.init_model, device)\n        apply_optc_gradient_shield(il_model, old_vocab_sizes, old_scale=0.35)\n\n    il_res = train_single_model(\n        il_model,\n        train_ds,\n        val_loader,\n        test_loader,\n        device=device,\n        output_dir=results_dir / "incremental",\n        epochs=args.epochs,\n        min_epochs=args.min_epochs,\n        batch_size=args.batch_size,\n        lr=args.lr,\n        patience=args.patience,\n        amp=amp,\n        seed=args.seed,\n        replay_ds=replay_ds,\n        replay_ratio=args.replay_ratio,\n        model_tag="incremental",\n        freeze_epochs=args.freeze_epochs,\n        loss_type=args.loss,\n        t_min=args.t_min,\n        t_max=args.t_max,\n    )\n    il_test_m = il_res["test"]\n    il_val_m = il_res["val"]\n    print(f"  Continual IL Test: F1={il_test_m[\'f1\']:.4f} | Prec={il_test_m[\'precision\']:.4f} | Rec={il_test_m[\'recall\']:.4f} | TP={il_test_m[\'tp\']}, FP={il_test_m[\'fp\']}, FN={il_test_m[\'fn\']}")\n\n    # -------------------------------------------------------------\n    # 4. Backward Transfer (BWT) Evaluation on OpTC Test Set\n    # -------------------------------------------------------------\n    bwt_metrics = {}\n    if args.optc_test_root and Path(args.optc_test_root).exists():\n        print("\\n" + "=" * 60)\n        print("EXPERIMENT 4: Backward Transfer (BWT) Evaluation on OpTC Test")\n        print("=" * 60)\n        optc_test_ds = PackedNetworkDataset(args.optc_test_root, "test")\n        optc_loader, _ = make_loader(optc_test_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed)\n\n        # Before IL (Base OpTC model) at frozen base threshold (0.9956)\n        b_true, b_prob, _ = predict(zs_model, optc_loader, device=device, amp=amp)\n        before_m = binary_metrics(b_true, b_prob, optc_frozen_thr)\n\n        # After IL (Continual Learning model) evaluated at Unified Threshold (Task 4.md Action 1)\n        # Note: In Domain-IL, the single calibrated threshold from combined validation is used without test leakage!\n        unified_thr = il_res["threshold"]\n        a_true, a_prob, _ = predict(il_model, optc_loader, device=device, amp=amp)\n        after_m_unified = binary_metrics(a_true, a_prob, unified_thr)\n        after_m_legacy = binary_metrics(a_true, a_prob, optc_frozen_thr)\n\n        bwt_f1 = after_m_unified["f1"] - before_m["f1"]\n        bwt_ap = after_m_unified["pr_auc"] - before_m["pr_auc"]\n        bwt_roc_auc = after_m_unified["roc_auc"] - before_m["roc_auc"]\n\n        bwt_metrics = {\n            "optc_before_il": before_m,\n            "optc_after_il": after_m_unified,\n            "optc_after_il_unified": after_m_unified,\n            "optc_after_il_legacy_thr": after_m_legacy,\n            "unified_threshold": float(unified_thr),\n            "bwt_f1": float(bwt_f1),\n            "bwt_f1_unified": float(bwt_f1),\n            "bwt_ap": float(bwt_ap),\n            "bwt_roc_auc": float(bwt_roc_auc),\n        }\n        dump_json(results_dir / "bwt_metrics.json", bwt_metrics)\n\n        print(f"  OpTC Test Before IL (Base Thr={optc_frozen_thr:.4f})  : F1={before_m[\'f1\']:.4f} | Prec={before_m[\'precision\']:.4f} | Rec={before_m[\'recall\']:.4f} | TP={before_m[\'tp\']}, FP={before_m[\'fp\']}")\n        print(f"  OpTC Test After IL  (Unified Thr={unified_thr:.4f}): F1={after_m_unified[\'f1\']:.4f} | Prec={after_m_unified[\'precision\']:.4f} | Rec={after_m_unified[\'recall\']:.4f} | TP={after_m_unified[\'tp\']}, FP={after_m_unified[\'fp\']}")\n        print(f"  OpTC Test After IL  (Legacy Thr={optc_frozen_thr:.4f}) : F1={after_m_legacy[\'f1\']:.4f} | Prec={after_m_legacy[\'precision\']:.4f} | Rec={after_m_legacy[\'recall\']:.4f} | TP={after_m_legacy[\'tp\']}, FP={after_m_legacy[\'fp\']}")\n        print(f"  Backward Transfer (BWT F1 Unified): {bwt_f1:+.4f} ({bwt_f1*100:+.2f}%)")\n        print(f"  Backward Transfer (BWT AP): {bwt_ap:+.4f} | BWT ROC-AUC: {bwt_roc_auc:+.4f}")\n\n    # -------------------------------------------------------------\n    # 5. Build comparison_metrics.csv (Matching Host ML Run 6 Standards)\n    # -------------------------------------------------------------\n    comp_rows = [\n        {\n            "Model Experiment": "ATLASv2 Scratch (Baseline)",\n            "Val AP": round(scratch_val_m["pr_auc"], 4),\n            "Val F1": round(scratch_val_m["f1"], 4),\n            "Threshold": round(scratch_res["threshold"], 4),\n            "Test AP": round(scratch_test_m["pr_auc"], 4),\n            "Test ROC-AUC": round(scratch_test_m["roc_auc"], 4),\n            "Test F1": round(scratch_test_m["f1"], 4),\n            "Test Precision": round(scratch_test_m["precision"], 4),\n            "Test Recall": round(scratch_test_m["recall"], 4),\n            "Test FPR": f"{scratch_test_m[\'fpr\']*100:.4f}%",\n            "TP": scratch_test_m["tp"],\n            "FP": scratch_test_m["fp"],\n            "FN": scratch_test_m["fn"],\n            "TN": scratch_test_m["tn"],\n        },\n        {\n            "Model Experiment": "OpTC -> ATLASv2 Continual Learning (IL)",\n            "Val AP": round(il_val_m["pr_auc"], 4),\n            "Val F1": round(il_val_m["f1"], 4),\n            "Threshold": round(il_res["threshold"], 4),\n            "Test AP": round(il_test_m["pr_auc"], 4),\n            "Test ROC-AUC": round(il_test_m["roc_auc"], 4),\n            "Test F1": round(il_test_m["f1"], 4),\n            "Test Precision": round(il_test_m["precision"], 4),\n            "Test Recall": round(il_test_m["recall"], 4),\n            "Test FPR": f"{il_test_m[\'fpr\']*100:.4f}%",\n            "TP": il_test_m["tp"],\n            "FP": il_test_m["fp"],\n            "FN": il_test_m["fn"],\n            "TN": il_test_m["tn"],\n        },\n        {\n            "Model Experiment": "OpTC Base Model (Zero-shot / Trước IL)",\n            "Val AP": round(zs_val_m["pr_auc"], 4),\n            "Val F1": round(zs_val_m["f1"], 4),\n            "Threshold": round(optc_frozen_thr, 4),\n            "Test AP": round(zs_test_m["pr_auc"], 4),\n            "Test ROC-AUC": round(zs_test_m["roc_auc"], 4),\n            "Test F1": round(zs_test_m["f1"], 4),\n            "Test Precision": round(zs_test_m["precision"], 4),\n            "Test Recall": round(zs_test_m["recall"], 4),\n            "Test FPR": f"{zs_test_m[\'fpr\']*100:.4f}%",\n            "TP": zs_test_m["tp"],\n            "FP": zs_test_m["fp"],\n            "FN": zs_test_m["fn"],\n            "TN": zs_test_m["tn"],\n        },\n    ]\n\n    comp_csv = results_dir / "comparison_metrics.csv"\n    with open(comp_csv, "w", newline="", encoding="utf-8") as f:\n        fieldnames = ["Model Experiment", "Val AP", "Val F1", "Threshold", "Test AP", "Test ROC-AUC", "Test F1", "Test Precision", "Test Recall", "Test FPR", "TP", "FP", "FN", "TN"]\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(comp_rows)\n    print(f"\\nSaved comparison metrics: {comp_csv}")\n\n    # -------------------------------------------------------------\n    # 6. Generate 3-Panel Confusion Matrix\n    # -------------------------------------------------------------\n    plot_3panel_confusion_matrix(\n        zs_test_m,\n        scratch_test_m,\n        il_test_m,\n        results_dir / "comparison_confusion_matrix.png",\n    )\n\n    # -------------------------------------------------------------\n    # 7. Traceback Analysis for False Negatives & False Positives\n    # -------------------------------------------------------------\n    run_traceback_analysis(\n        test_ds,\n        il_res["test_ids"],\n        il_res["test_true"],\n        il_res["test_prob"],\n        il_res["threshold"],\n        results_dir / "traceback_fn_fp.csv",\n    )\n\n    print("\\n" + "=" * 80)\n    print("ALL BENCHMARK RUNS, TRACEBACK, AND VISUALIZATIONS COMPLETE!")\n    print("=" * 80)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

print('Wrote training modules to:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))


## DataLoader sanity
Một batch trước khi full training.


In [ ]:
from dataset import PackedNetworkDataset, ShardAwareBatchSampler, collate_sequence_batch
from torch.utils.data import DataLoader
ds = PackedNetworkDataset(DATASET_ROOT, 'train', cache_shards=1)
sampler = ShardAwareBatchSampler(ds, 8, shuffle=False)
loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, collate_fn=collate_sequence_batch)
b = next(iter(loader))
print('Train samples:', len(ds))
print('x_cat shape:  ', tuple(b['x_cat'].shape))
print('x_num shape:  ', tuple(b['x_num'].shape))
print('x_bin shape:  ', tuple(b['x_bin'].shape))
print('delta_t shape:', tuple(b['delta_t_log1p'].shape))
print('lengths:      ', b['lengths'].tolist())
print('app_family IDs in batch:', sorted(set(b['x_cat'][:, :, 0].unique().tolist()))[:10])


## Train BiLSTM baseline
`BCEWithLogitsLoss(pos_weight=TRAIN_NEG/TRAIN_POS)`, early stopping theo VAL PR-AUC, threshold maximize F1 trên VAL.


In [ ]:
# Locate OpTC Pretrained Checkpoint & Experience Replay Buffer
INIT_MODEL = None
candidates = [
    DATASET_ROOT / 'pretrained_optc_model/best_model.pt',
    Path('/kaggle/input/datasets/ngkimngn/dl-il-atlas/pretrained_optc_model/best_model.pt'),
    Path('/kaggle/input/dl-il-atlas/pretrained_optc_model/best_model.pt'),
    Path('/kaggle/input/pretrained_optc_model/best_model.pt'),
    Path('/kaggle/input/optc_network_dl/pretrained_optc_model/best_model.pt'),
]
for cand in candidates:
    if cand.exists():
        INIT_MODEL = cand
        break

REPLAY_BUFFER = DATASET_ROOT / 'optc_replay_buffer.npz'
if not REPLAY_BUFFER.exists():
    fallback_replay = Path('/kaggle/input/datasets/ngkimngn/dl-il-atlas/optc_replay_buffer.npz')
    if fallback_replay.exists():
        REPLAY_BUFFER = fallback_replay

OPTC_TEST_DIR = DATASET_ROOT / 'optc_test'
if not OPTC_TEST_DIR.exists():
    fallback_test = Path('/kaggle/input/datasets/ngkimngn/dl-il-atlas/optc_test')
    if fallback_test.exists():
        OPTC_TEST_DIR = fallback_test

OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')

cmd_parts = [
    'python',
    str(WORK_CODE / 'train_incremental.py'),
    '--dataset-root', f"'{DATASET_ROOT}'",
    '--output-dir', f"'{OUTPUT_DIR}'",
    '--epochs', '10',
    '--min-epochs', '3',
    '--loss', 'focal',
    '--t-min', '0.25',
    '--t-max', '0.75',
    '--batch-size', '32',
    '--lr', '0.0002',
    '--weight-decay', '0.0001',
    '--patience', '4',
    '--seed', '42',
    '--run-benchmark',
]

if INIT_MODEL is not None and INIT_MODEL.exists():
    cmd_parts.extend(['--init-model', f"'{INIT_MODEL}'"])
    print(f'[PRETRAINED CHECKPOINT] Using OpTC base weights: {INIT_MODEL}')

if REPLAY_BUFFER.exists():
    cmd_parts.extend(['--replay-buffer', f"'{REPLAY_BUFFER}'", '--replay-ratio', '0.05'])
    print(f'[EXPERIENCE REPLAY] Using OpTC buffer: {REPLAY_BUFFER} (ratio: 5% - MalFSCIL 2024)')

if OPTC_TEST_DIR.exists():
    cmd_parts.extend(['--optc-test-root', f"'{OPTC_TEST_DIR}'"])
    print(f'[BACKWARD TRANSFER] Evaluating on OpTC benchmark test set: {OPTC_TEST_DIR}')

cmd = ' '.join(cmd_parts)
print('\nRunning Command:\n', cmd)

import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}')


## Final metrics


In [ ]:
import pandas as pd
from IPython.display import display

comp_file = OUTPUT_DIR / 'results' / 'comparison_metrics.csv'
bwt_file = OUTPUT_DIR / 'results' / 'bwt_metrics.json'

print('=' * 80)
print('BILSTM BENCHMARK COMPARISON ON ATLASv2 (DOMAIN-INCREMENTAL LEARNING)')
print('=' * 80)

if comp_file.exists():
    df_comp = pd.read_csv(comp_file)
    display(df_comp)
else:
    print('Warning: comparison_metrics.csv not found.')

if bwt_file.exists():
    bwt = json.loads(bwt_file.read_text())
    print('\n' + '=' * 80)
    print('BACKWARD TRANSFER (BWT) ON OPTC TEST BENCHMARK')
    print('=' * 80)
    after_il = bwt.get('optc_after_il', bwt.get('optc_after_il_unified', {}))
    bwt_f1 = bwt.get('bwt_f1', bwt.get('bwt_f1_unified', 0.0))
    bwt_ap = bwt.get('bwt_ap', 0.0)
    print(f"OpTC Test Before IL : F1={bwt['optc_before_il']['f1']:.4f} | Prec={bwt['optc_before_il']['precision']:.4f} | Rec={bwt['optc_before_il']['recall']:.4f} | TP={bwt['optc_before_il']['tp']}, FP={bwt['optc_before_il']['fp']}")
    print(f"OpTC Test After IL  : F1={after_il.get('f1', 0.0):.4f} | Prec={after_il.get('precision', 0.0):.4f} | Rec={after_il.get('recall', 0.0):.4f} | TP={after_il.get('tp', 0)}, FP={after_il.get('fp', 0)}")
    print(f"Backward Transfer (BWT F1): {bwt_f1:+.4f} ({bwt_f1*100:+.2f}%)")
    print(f"Backward Transfer (BWT AP): {bwt_ap:+.4f}")


## Traceback Analysis (Phân tích chi tiết False Negatives & False Positives)
Trích xuất thông tin nguồn của các cửa sổ bị nhầm lẫn trên tập Test của ATLASv2 (hostname, scenario, timestamp, flow count).


In [ ]:
trace_file = OUTPUT_DIR / 'results' / 'traceback_fn_fp.csv'
if trace_file.exists():
    df_trace = pd.read_csv(trace_file)
    print(f'Total Error Samples on Test Set: {len(df_trace)}')
    if len(df_trace) > 0:
        display(df_trace)
    else:
        print('Hoàn hảo: 0 FN, 0 FP trên tập Test!')
else:
    print('traceback_fn_fp.csv not found.')


## So sánh 3-Panel Confusion Matrix
So sánh trực quan ma trận nhầm lẫn giữa (1) Zero-shot, (2) Scratch Baseline, và (3) Continual Learning (IL + ER).


In [ ]:
from IPython.display import Image
cm_img = OUTPUT_DIR / 'results' / 'comparison_confusion_matrix.png'
if cm_img.exists():
    display(Image(filename=str(cm_img)))
else:
    print('comparison_confusion_matrix.png not found.')


## Zip training outputs


In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{RUN_NAME}', 'zip', OUTPUT_DIR)
print('Created:', archive)
